In [ ]:
#!/usr/bin/env python3
"""
T3 -- GNN-oracle reliability on MODELED structures (Kaggle, GPU).
BLOCKED in-sandbox: needs a structure predictor (ESMFold or AlphaFold-class,
GPU) and a GNN training stack (torch + torch_geometric) -- none available in
this project's dev sandbox (confirmed: `pip install torch` fails with
"No space left on device"). See ../PREREGISTRATION.md for the locked
threshold this script must evaluate against BEFORE trusting any number it
produces:
    PASS-for-the-proposal: AUROC on modeled structures >= 0.65 AND within
    0.10 AUROC of the crystal-structure number.
    FAIL: AUROC on modeled structures <= 0.60, OR a drop of >0.15 AUROC
    from crystal to modeled.

Data
----
Real TCR:peptide:MHC ternary complexes: STCRDab's own web portal was
unreachable from this project's sandbox this session (confirmed via direct
curl -- naga-www-prod.stats.ox.ac.uk timed out / returned empty). Substitute:
RCSB PDB's own search API (search.rcsb.org), which STCRDab itself curates
FROM -- same underlying real crystal structures, just queried directly.
`../data/processed/rcsb_tcr_pmhc_candidates.csv` (built by
../fetch_rcsb_tcr_structures.py, run in-sandbox -- see that script) has the
candidate PDB ID list; this script downloads the actual coordinate files on
Kaggle (network available there) and extracts CDR3-peptide interface residue
contacts as the GNN's input graphs.

Pipeline
--------
1. Download PDB structures for the candidate ID list (Kaggle has network).
2. Parse chains, identify TCR alpha/beta CDR3 loops (IMGT numbering) and the
   bound peptide; build a residue-contact graph (nodes = interface residues,
   edges = spatial contacts within 5A, node features = residue identity +
   one-hot chain role).
3. Label: binder (this complex IS a real solved TCR:pMHC complex -> positive)
   vs non-binder (decoy: computationally re-dock a DIFFERENT peptide from
   the dataset onto the same TCR's modeled structure, or use a structurally
   plausible but non-cognate peptide-MHC pairing as the negative class --
   document exactly which decoy strategy is used, since decoy quality
   determines whether this test is meaningful at all).
4. Train a small message-passing GNN (e.g. 3-layer GCN/GAT, torch_geometric)
   on crystal-structure graphs, held-out test split at the COMPLEX level
   (not residue level) to avoid leakage.
5. Evaluate the SAME trained GNN on (a) held-out crystal structures (upper
   bound) and (b) ESMFold-modeled structures of the SAME held-out complexes
   (the actual production condition -- the generator's outputs will never be
   crystal structures).
6. Report AUROC for (a) and (b) side by side, apply the locked threshold.

Usage (Kaggle: New Notebook -> enable Internet + GPU -> paste/upload -> Run All)
    pip install torch torch_geometric biopython fair-esm
"""
from __future__ import annotations
import json, os
from pathlib import Path

IS_KAGGLE = Path("/kaggle").exists()
REPO = Path("/kaggle/working") if IS_KAGGLE else Path(__file__).resolve().parent.parent
PROC = REPO / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)

IN_CANDIDATES = PROC / "rcsb_tcr_pmhc_candidates.csv"
OUT_REPORT = PROC / "T3_gnn_oracle_report.json"

CROP_RADIUS_A = 5.0
BAR_MIN_MODELED_AUROC = 0.65
BAR_MAX_CRYSTAL_MODELED_DROP = 0.15
BAR_MAX_CRYSTAL_MODELED_WINDOW = 0.10  # "within 0.10" pass condition


def _fetch_candidates_via_rcsb(out_csv: Path, max_rows: int = 2000):
    """Self-fetch fallback: same query fetch_rcsb_tcr_structures.py runs
    in-sandbox, inlined here so this notebook doesn't depend on a manually
    staged CSV. Kaggle has network (confirmed by the pip installs in this
    same run), so this is safe to run here too."""
    import json, time, urllib.request
    import pandas as pd

    SEARCH_URL = "https://search.rcsb.org/rcsbsearch/v2/query"
    GRAPHQL_URL = "https://data.rcsb.org/graphql"

    def search_ids():
        query = {
            "query": {
                "type": "group", "logical_operator": "and",
                "nodes": [
                    {"type": "terminal", "service": "full_text",
                     "parameters": {"value": "T cell receptor"}},
                    {"type": "terminal", "service": "full_text",
                     "parameters": {"value": "major histocompatibility complex"}},
                ],
            },
            "return_type": "entry",
            "request_options": {"return_all_hits": False, "paginate": {"start": 0, "rows": max_rows}},
        }
        req = urllib.request.Request(SEARCH_URL, data=json.dumps(query).encode(),
                                      headers={"Content-Type": "application/json"})
        with urllib.request.urlopen(req, timeout=30) as r:
            res = json.load(r)
        return [x["identifier"] for x in res.get("result_set", [])], res.get("total_count", 0)

    def fetch_metadata_batch(ids_batch):
        gql = """
        query($ids: [String!]!) {
          entries(entry_ids: $ids) {
            rcsb_id
            struct { title }
            rcsb_entry_info { deposited_polymer_entity_instance_count resolution_combined }
          }
        }
        """
        body = json.dumps({"query": gql, "variables": {"ids": ids_batch}}).encode()
        req = urllib.request.Request(GRAPHQL_URL, data=body, headers={"Content-Type": "application/json"})
        with urllib.request.urlopen(req, timeout=30) as r:
            res = json.load(r)
        rows = []
        for e in res.get("data", {}).get("entries", []) or []:
            if e is None:
                continue
            info = e.get("rcsb_entry_info") or {}
            res_list = info.get("resolution_combined") or []
            rows.append({
                "pdb_id": e["rcsb_id"],
                "title": (e.get("struct") or {}).get("title"),
                "n_chains": info.get("deposited_polymer_entity_instance_count"),
                "resolution": res_list[0] if res_list else None,
            })
        return rows

    ids, total = search_ids()
    print(f"[self-fetch] {len(ids)} candidate IDs fetched (RCSB reports {total} total full-text hits)")
    rows = []
    BATCH = 50
    for i in range(0, len(ids), BATCH):
        batch = ids[i:i + BATCH]
        try:
            rows.extend(fetch_metadata_batch(batch))
        except Exception as e:
            print(f"[warn] batch {i} failed: {e}")
        time.sleep(0.2)
    df = pd.DataFrame(rows)
    df_filtered = df[df["n_chains"].fillna(0) >= 4].copy()
    print(f"[self-fetch] {len(df_filtered)}/{len(df)} entries have >=4 polymer chains")
    df_filtered.to_csv(out_csv, index=False)
    return df_filtered


def main():
    import pandas as pd
    if not IN_CANDIDATES.exists():
        # Prefer a manually staged copy (from ../fetch_rcsb_tcr_structures.py,
        # run in-sandbox) if present; otherwise self-fetch here, since Kaggle
        # has network access in this session (confirmed by this run's own
        # pip installs succeeding). This makes the notebook runnable
        # standalone without a separate upload step.
        print(f"[info] {IN_CANDIDATES} not found -- self-fetching from RCSB "
              f"(no pre-staged CSV needed; skip this by uploading the file "
              f"from ../data/processed/ as a Kaggle Dataset input instead).")
        cands = _fetch_candidates_via_rcsb(IN_CANDIDATES)
    else:
        cands = pd.read_csv(IN_CANDIDATES)
    print(f"[load] {len(cands)} candidate PDB IDs")

    # NOTE TO IMPLEMENTER (Eric / whoever runs this on Kaggle):
    # This scaffold defines the exact I/O contract and the locked pass/fail
    # bar. The structure-download / IMGT-CDR3-parsing / graph-construction /
    # ESMFold-modeling / GNN-training steps are GPU- and dependency-heavy
    # (torch_geometric + biopython + fair-esm's ESMFold, ~a few GB of
    # weights) and were deliberately NOT stubbed with fake logic here --
    # writing untested placeholder GNN code would risk silently producing a
    # number that looks real but isn't. Fill in steps 1-5 from the docstring
    # above using this candidate list as input, then call `write_report()`
    # below with the two real AUROC numbers.
    raise NotImplementedError(
        "Structure download + CDR3/interface parsing + ESMFold modeling + GNN train/eval "
        "not implemented in this scaffold -- GPU + heavy deps required, do this on Kaggle. "
        "See docstring for the exact pipeline and REPO/PROC paths already wired up.")


def write_report(auroc_crystal: float, auroc_modeled: float, n_test_complexes: int, decoy_strategy: str):
    """Call this once the real AUROC numbers are in hand."""
    drop = auroc_crystal - auroc_modeled
    within_window = abs(drop) <= BAR_MAX_CRYSTAL_MODELED_WINDOW
    passes = (auroc_modeled >= BAR_MIN_MODELED_AUROC) and within_window
    fails = (auroc_modeled <= 0.60) or (drop > BAR_MAX_CRYSTAL_MODELED_DROP)
    verdict = "PASS" if passes else ("FAIL" if fails else "AMBIGUOUS -- report raw numbers, don't force a call")
    report = {
        "n_test_complexes": n_test_complexes,
        "decoy_strategy": decoy_strategy,
        "auroc_crystal_structures": auroc_crystal,
        "auroc_modeled_structures": auroc_modeled,
        "crystal_to_modeled_drop": drop,
        "bars": {"min_modeled_auroc": BAR_MIN_MODELED_AUROC,
                 "max_crystal_modeled_window": BAR_MAX_CRYSTAL_MODELED_WINDOW,
                 "fail_drop_threshold": BAR_MAX_CRYSTAL_MODELED_DROP},
        "verdict": verdict,
    }
    with open(OUT_REPORT, "w") as f:
        json.dump(report, f, indent=2)
    print(f"[done] wrote {OUT_REPORT}: {verdict}")


if __name__ == "__main__":
    main()
